# Лабораторная работа 4

## Классификация цветов автомобилей

План:

1. Проверить структуру датасета.
2. Сравнить две предобученные модели.
3. Сравнить их с собственной CNN, обученной с нуля.
4. Оценить качество по `F1_macro` и сделать вывод.

In [5]:
%matplotlib inline

import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

project_root = Path.cwd()
artifacts_root = project_root / "artifacts" / "lab4"
manifest_path = project_root / "data" / "prepared_lab4" / "manifest.json"

with manifest_path.open(encoding="utf-8") as file:
    manifest = json.load(file)

summary = pd.read_csv(artifacts_root / "results_summary.csv")
distribution = pd.read_csv(artifacts_root / "class_distribution.csv")
imagenet_report = pd.read_csv(artifacts_root / "vit_b16_imagenet_report.csv")
swag_report = pd.read_csv(artifacts_root / "vit_b16_swag_report.csv")
scratch_report = pd.read_csv(artifacts_root / "from_scratch_report.csv")
scratch_history = pd.read_csv(artifacts_root / "scratch_history.csv")

with (artifacts_root / "vit_b16_imagenet_metrics.json").open(encoding="utf-8") as file:
    imagenet_metrics = json.load(file)

with (artifacts_root / "vit_b16_swag_metrics.json").open(encoding="utf-8") as file:
    swag_metrics = json.load(file)

with (artifacts_root / "scratch_metrics.json").open(encoding="utf-8") as file:
    scratch_metrics = json.load(file)

with (artifacts_root / "scratch_stats.json").open(encoding="utf-8") as file:
    scratch_stats = json.load(file)

with (artifacts_root / "conclusion.json").open(encoding="utf-8") as file:
    conclusion = json.load(file)

display(Markdown(f"**Классы:** {', '.join(manifest['classes'])}"))
display(Markdown(f"**Размеры сплитов:** train = {distribution.query('split == \"train\"')['count'].sum()}, val = {distribution.query('split == \"val\"')['count'].sum()}, test = {distribution.query('split == \"test\"')['count'].sum()}"))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/maxim-bovt/PycharmProjects/CV/ITMO-cv-labs-bovt/data/prepared_lab4/manifest.json'

## Распределение данных

Датасет сбалансирован по цветам, поэтому сравнение моделей получается корректным.

In [ ]:
distribution_pivot = distribution.pivot(index="class_name", columns="split", values="count")
display(distribution_pivot)
display(Image(filename=str(artifacts_root / "class_distribution.png")))

## Модели

Использованы три простых варианта:

- `vit_b16_imagenet` — признаки `ViT-B/16` с весами `ImageNet` и линейный классификатор.
- `vit_b16_swag` — признаки `ViT-B/16` с весами `SWAG` и линейный классификатор.
- `scratch_cnn` — компактная CNN, обученная с нуля.

In [ ]:
summary_rounded = summary.copy()
summary_rounded["test_accuracy"] = summary_rounded["test_accuracy"].round(4)
summary_rounded["test_f1_macro"] = summary_rounded["test_f1_macro"].round(4)
display(summary_rounded)

metrics_frame = pd.DataFrame([
    imagenet_metrics,
    swag_metrics,
    scratch_metrics,
])
display(metrics_frame)

## Отчёты по классам

Ниже видно, что лучше всего модели определяют `Red`, а хуже всего обычно определяют `Grey`.

In [ ]:
report_columns = ["label", "precision", "recall", "f1-score", "support"]

display(Markdown("### ViT-B/16 ImageNet"))
display(imagenet_report[report_columns])

display(Markdown("### ViT-B/16 SWAG"))
display(swag_report[report_columns])

display(Markdown("### Scratch CNN"))
display(scratch_report[report_columns])

## Визуализация результатов

Матрицы ошибок и кривая обучения помогают быстро увидеть, где модели ошибаются чаще всего.

In [ ]:
display(Markdown("### ViT-B/16 ImageNet"))
display(Image(filename=str(artifacts_root / "vit_b16_imagenet_confusion.png")))

display(Markdown("### ViT-B/16 SWAG"))
display(Image(filename=str(artifacts_root / "vit_b16_swag_confusion.png")))

display(Markdown("### Scratch CNN"))
display(Image(filename=str(artifacts_root / "from_scratch_confusion.png")))
display(Image(filename=str(artifacts_root / "scratch_learning_curve.png")))

## Своя модель с нуля

Архитектура специально сделана компактной, потому что датасет небольшой и задача не требует глубокой сети с большим числом параметров.

In [ ]:
import torch
from torch import nn

class ScratchClassifier(nn.Module):
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        outputs = self.features(inputs)
        return self.classifier(outputs)

scratch_model = ScratchClassifier(num_classes=len(manifest["classes"]))
scratch_model

In [ ]:
display(pd.DataFrame([scratch_stats]))
display(scratch_history.tail())

## Вывод

Лучший результат показала модель `vit_b16_swag`.

`F1_macro` здесь важен тем, что усредняет качество по всем цветам и не позволяет одному классу слишком сильно влиять на итоговую оценку.

In [ ]:
display(pd.DataFrame([conclusion]))

best_gap = float(summary.iloc[0]["test_f1_macro"] - summary.iloc[-1]["test_f1_macro"])
display(Markdown(f"**Разница между лучшей и худшей моделью по F1_macro:** {best_gap:.4f}"))